# Lesson 7 | The first RTL neuron

Today combines state/clock, Boolean logic, and RTL:
> **Split one neuron update into a combinational path and a sequential update.**

To avoid silently choosing unfrozen LIF/fixed-point details, this lesson uses a **teaching integrate-and-fire neuron** that reuses Lesson 4's accumulator + threshold + reset rule. It is not formal `MOD-003`.


## 1. Lesson contract

Each cycle: read old `membrane_v` → `candidate = membrane_v + input_current` → `candidate >= threshold` produces spike → spike selects reset, otherwise candidate → the clock edge stores next state.

Vectors avoid overflow. Formal leak/rounding/overflow remain controlled by RMD-002/RMD-003.


In [ ]:
def tutorial_if_step(state, input_current, threshold, reset_value=0):
    candidate = state + input_current
    spike = candidate >= threshold
    return (reset_value if spike else candidate), spike, candidate

state = 0
for cycle, current in enumerate([1,1,1,1,2,2]):
    nxt, spike, candidate = tutorial_if_step(state,current,4)
    print(f'cycle={cycle}: state={state}, input={current}, candidate={candidate}, spike={spike}, next={nxt}')
    state = nxt


## 2. Structure

```mermaid
flowchart LR
 REG["membrane_v register"] --> ADD["adder"]
 IN["input_current"] --> ADD
 ADD --> C["candidate"]
 C --> CMP[">= threshold"]
 TH["threshold"] --> CMP
 CMP --> MUX["reset or candidate"]
 C --> MUX
 R["reset_value"] --> MUX
 MUX --> REG
 CLK["clock edge"] -.-> REG
```


## 3. `always_comb` and `always_ff`

`always_comb` describes combinational next-state computation; `always_ff` stores state at the clock edge. `candidate_ext` / `spike_next` / `next_v` are current-cycle calculations; `membrane_v` is retained state.


## 4. Complete teaching RTL

```systemverilog
module tutorial_if_neuron #(
    parameter int WIDTH = 8
) (
    input  logic clk,
    input  logic rst_n,
    input  logic signed [WIDTH-1:0] input_current,
    input  logic signed [WIDTH-1:0] threshold,
    input  logic signed [WIDTH-1:0] reset_value,
    output logic signed [WIDTH-1:0] membrane_v,
    output logic spike
);
    logic signed [WIDTH:0] candidate_ext;
    logic signed [WIDTH-1:0] next_v;
    logic spike_next;

    always_comb begin
        candidate_ext = $signed({membrane_v[WIDTH-1], membrane_v})
                      + $signed({input_current[WIDTH-1], input_current});
        spike_next = candidate_ext >= $signed({threshold[WIDTH-1], threshold});
        next_v = spike_next ? reset_value : candidate_ext[WIDTH-1:0];
    end

    always_ff @(posedge clk) begin
        if (!rst_n) begin
            membrane_v <= reset_value;
            spike <= 1'b0;
        end else begin
            membrane_v <= next_v;
            spike <= spike_next;
        end
    end
endmodule
```


## 5. Why is candidate one bit wider?

Adding two WIDTH-bit signed numbers may need one extra bit, so the sum uses `candidate_ext[WIDTH:0]`. A non-spiking value beyond WIDTH would truncate here; our vectors avoid that. Formal RTL must obey the future frozen overflow policy.


## 6. Try It

Before an edge: `membrane_v=3, input=1, threshold=4, reset=0`. Write candidate, spike_next, next_v, then post-edge membrane_v/spike.


## 7. AI Task / Human Check

Ask AI to map each of the five contract rules to exact RTL lines without changing the contract.

Without AI, identify the combinational path and stored state, and explain how `candidate=4` and post-edge `membrane_v=0` can both be correct.


## 8. Engineering Handoff / Project Trace

`rtl/learning/tutorial_if_neuron.sv` is a teaching precursor to RMD-004; formal `rtl/neuron/lif_neuron_engine.sv` still waits for v0 semantics and the fixed-point contract.

- Lesson: `LSN-007`
- Mapping: `RMD-004` teaching precursor
- Formal `MOD-003`: intentionally not created


## 9. Exit Ticket

You can derive a combinational-path-plus-register structure from the contract and identify both parts in RTL. Next lesson adds no neuron behavior; it verifies this one.
